In [ ]:
# import libraries
try:
  # %tensorflow_version only exists in Colab.
  !pip install tf-nightly
except Exception:
  pass
import tensorflow as tf
import pandas as pd
from tensorflow import keras
!pip install tensorflow-datasets
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:

# Read the training and testing data
train_df = pd.read_csv(
    train_file_path,
    sep='\t',
    header=None,
    names=['label', 'message']
)

test_df = pd.read_csv(
    test_file_path,
    sep='\t',
    header=None,
    names=['label', 'message']
)

# Convert labels to numerical values
train_df['label'] = train_df['label'].map({'ham': 0, 'spam': 1})
test_df['label'] = test_df['label'].map({'ham': 0, 'spam': 1})

# Prepare text data
train_messages = train_df['message'].astype(str).values
train_labels = train_df['label'].values

test_messages = test_df['message'].astype(str).values
test_labels = test_df['label'].values

print(train_df.head())
print("Training samples:", len(train_df))
print("Testing samples:", len(test_df))


In [ ]:

# Import layers
from tensorflow.keras import layers

# Create a text vectorization layer
max_tokens = 10000
vectorizer = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode='tf_idf',
    ngrams=2
)

# Learn vocabulary from training messages
vectorizer.adapt(train_messages)

# Build the neural network
model = tf.keras.Sequential([
    vectorizer,
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train the model
history = model.fit(
    train_messages,
    train_labels,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# Evaluate the model
loss, accuracy = model.evaluate(
    test_messages,
    test_labels,
    verbose=0
)

print(f"Test accuracy: {accuracy:.4f}")

In [ ]:

def predict_message(pred_text):
    prediction = model.predict(
        tf.constant([pred_text]),
        verbose=0
    )[0][0]

    probability = float(prediction)

    if probability >= 0.25:
        label = "spam"
    else:
        label = "ham"

    return [probability, label]


pred_text = "how are you doing today?"
prediction = predict_message(pred_text)
print(prediction)


In [ ]:

test_messages = [
    "how are you doing today",
    "sale today! to stop texts call 98912460324",
    "i don't want to go. can we try it a different day? available sat",
    "our new mobile video service is live. just install on your phone to start watching.",
    "you have won £1000 cash! call to claim your prize.",
    "i'll bring it tomorrow. don't forget the milk.",
    "wow, is your arm alright. that happened to me one time too"
]

for msg in test_messages:
    result = predict_message(msg)
    print(f"Message: {msg}")
    print(f"Prediction: {result[1]} | Spam probability: {result[0]:.4f}")
    print("-" * 50)


In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
